# 基礎編２
このノートブックでは、ウォレットを作成し、ブロックチェーンにユーザ登録する例を示します。

## 準備
実行の準備を行います。

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { chainID, domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc } = await import('../lib/notebook-util.mjs');

## ウォレットの新規作成
ウォレットはブロックチェーンに接続せずにオフチェーンで作成できます。  

ウォレットをメモリ上に作成します。

In [2]:
// 'es'は作成するウォレットファイルのタイプ
var newWalletPassword = 'p1616ant10';
var newWfstr = await api.createWalletFile('user123', newWalletPassword, 'es');

作成したウォレットをファイルに保存します。  
ファイルに保存される情報は、パスワードで暗号化されています。

In [3]:
var fs = await import('node:fs');
var path = await import('node:path');
var { default: package_root } = await import('../lib/get-package-root.mjs'); // samplesフォルダ
fs.writeFileSync(path.join(package_root, 'etc', 'wallet-user123.json'), newWfstr, 'utf8');

ウォレットをパスワードで開きます。  

In [4]:
var newWf = await api.parseWalletFile(newWfstr);
var newUserWallet = await api.unlockWalletFile(newWf, newWalletPassword);
var newUserAddress = newUserWallet.address;
console.log('address:', newUserAddress);

address: eetYxbxipRU6o2DTfLvGZ3cth4WJZh


パスワードが間違っていると開けません。

In [5]:
var newUserUw = await api.unlockWalletFile(newWf, 'wrong password');

Error: invalid password

## ユーザの新規作成とウォレットの紐づけ
ウォレットの紐づけには、ウォレットアドレスのみが必要です。

システムコントラクト'c1create'を使って、ブロックチェーン上にユーザを作成します。  
作成したユーザのIDを取得します。

In [6]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'user', domain });
var newUserId = resp.value;
console.log('new user ID:', newUserId);

new user ID: u11897627


システムコントラクト'c1update'を使って、作成したユーザにウォレットを紐づけます。

In [7]:
var resp = await rpc.call(adminWallet, 'c1update', { id: newUserId, prop: 'add wallets', value: [ newUserAddress ] });
console.log(resp);

{
  txno: 701649,
  txid: 'xfZpj6KvCrwCtEfoHNf82qQVbKg4fFrQbmkSzytrqrHY5',
  status: 'ok',
  value: null
}


## 確認

確認のため、新しいウォレットを使って、自ユーザをブロックチェーンに問い合わせます。  
newUserIdと同じユーザIDが返ってくることを確かめます。

In [8]:
var resp = await rpc.call(newUserWallet, 'c1query', { type: 'a_wallet' });
if (resp.value.user[0] === newUserId) console.log('OK'); else console.log('NG');
console.log(resp);

OK
{
  txid: 'xsdmn5kxdYHrme52N3qp2LZyikv3mSbrd3o8QPXZVvvhq',
  status: 'ok',
  value: { user: [ 'u11897627', 'u11897627@d93319890' ] }
}


確認のため、ユーザの情報をブロックチェーンに問い合わせます。  
ウォレットアドレスが登録されていることを確かめます。

In [9]:
var resp = await rpc.call(newUserWallet, 'c1query', { type: 'a_user', id: newUserId });
if (resp.value.wallets[0] === newUserAddress) console.log('OK'); else console.log('NG');
console.log(resp);

OK
{
  txid: 'xEuHwBQALd58akgJaQugWQUwBX8ujUuSv9jCsYQtxLqr2',
  status: 'ok',
  value: {
    id: [ 'u11897627', 'u11897627@d93319890' ],
    frozen: null,
    name: '',
    domain: 'd93319890',
    description: '',
    multisig: 1,
    a_txno: 701648,
    c_txno: 701648,
    num_transactions: 0,
    last_active: 1753420484707,
    created_at: 1753420484707,
    wallets: [ 'eetYxbxipRU6o2DTfLvGZ3cth4WJZh' ],
    callable_to: [ 'all' ],
    disclosed_to: [],
    groups: [],
    managed: false
  }
}


確認のため、新しいウォレットを使ってトランザクションを発行し、そのcallerがnewUserIdと一致していることを確かめます。

In [10]:
var resp = await rpc.call(newUserWallet, 'c1update', { id: newUserId, prop: 'name', value: 'user123' });
var resp = await rpc.call(newUserWallet, 'c1query', { type: 'a_transaction', id: resp.txid });
if (resp.value.caller[0] === newUserId) console.log('OK'); else console.log('NG');
console.log(resp);

OK
{
  txid: 'x9L7MCszLB2aJBu9xQERjJg5zDQkouC8FMGTpfZrqkisLB',
  status: 'ok',
  value: {
    blkno: 0,
    time: 1753420486559,
    txid: 'x5LGwE6GWacM9nCVkkcNYJc7awc2kdS8GovfgDGTPNvH6',
    addr: 'eetYxbxipRU6o2DTfLvGZ3cth4WJZh',
    caller: [ 'u11897627', 'u11897627@d93319890' ],
    callee: [ 'c1update', 'c1update@default' ],
    status: 'denied',
    pack64: 'BQAAAAEAAAD8AAAAAgAAAEAAAABAAAAAA3siY29udHJhY3QiOiJjMXVwZGF0ZSIsImFyZ3MiOnsiaWQiOiJ1MTE4OTc2MjciLCJwcm9wIjoibmFtZSIsInZhbHVlIjoidXNlcjEyMyJ9LCJhZGRyIjoiZWV0WXhieGlwUlU2bzJEVGZMdkdaM2N0aDRXSlpoIiwiZGVhZGxpbmUiOjE3NTM0MjA1ODY0NDQsIm9yYWNsZSI6IkpLU2FDRlM3QWZIa3hRR1kiLCJibG9ja3JlZiI6eyJubyI6NzMyMjksImhhc2giOiJ3TlJaaXBpRUdPUytHcHg5aEh2SzRFN0ZjR3RTYldJZk05RU4vWENRWGdzPSJ9fWVzIIj8Xk4cZURAxBnp2VanGEljODzQm//DNcLBeJeIpZmQLw3wEGRpwf7q21hlJLWZoJrlBP2hAGKfpnJZgwLs+xeWz5NaXBf/2SJW/8H0IZ/UUPKwiqw8MaRFDyFvwS30PklXDvBRYPkjQm+Z1zXq/jRJK4sqJUEkMRiObyEK7sg=',
    txno: 701650,
    caller_txno: 0,
    argstr: '{"id":"u11897627","prop":"name","va

## 二個目のウォレットの紐づけ
ユーザに複数のウォレットを紐づけることができます。

二個目のウォレットを作成します。  
例として区別しやすくするため、一個目とは違う種類のウォレットとします。  
（ウォレットアドレスの一文字目が異なる）

In [11]:
var anotherWalletPassword = 'p1616ant11';
var anotherWfstr = await api.createWalletFile('user124', anotherWalletPassword, 'r', chainID);
var anotherWf = await api.parseWalletFile(anotherWfstr);
var anotherUserWallet = await api.unlockWalletFile(anotherWf, anotherWalletPassword);
var anotherUserAddress = anotherUserWallet.address;
console.log('address:', anotherUserAddress);

address: rUmxPNLuDAtJb67LT8XNcidz6y82Bk


二個目のウォレットをユーザに紐づけます。

In [12]:
var resp = await rpc.call(adminWallet, 'c1update', { id: newUserId, prop: 'add wallets', value: [ anotherUserAddress ] });
console.log(resp);

{
  txno: 701651,
  txid: 'xeEjsDxF6nQsicEdgh58ZMKWxvRYuDEpfM2tDo5JBXEKz',
  status: 'ok',
  value: null
}


## 確認

確認のため、二個目のウォレットを使って、自ユーザをブロックチェーンに問い合わせます。  
newUserIdと同じユーザIDが返ってくることを確かめます。

In [13]:
var resp = await rpc.call(anotherUserWallet, 'c1query', { type: 'a_wallet' });
if (resp.value.user[0] === newUserId) console.log('OK'); else console.log('NG');
console.log(resp);

OK
{
  txid: 'xLLn8EJqbfwigtNXJNv8LSXuDUNQJidqodyWkXSK6howq',
  status: 'ok',
  value: { user: [ 'u11897627', 'u11897627@d93319890' ] }
}


確認のため、ユーザの情報をブロックチェーンに問い合わせます。  
ウォレットアドレスが２つ登録されていることを確かめます。

In [14]:
var resp = await rpc.call(anotherUserWallet, 'c1query', { type: 'a_user', id: newUserId });
if (resp.value.wallets[1] === anotherUserAddress) console.log('OK'); else console.log('NG');
console.log(resp);

OK
{
  txid: 'xef2LQz8UkXH5D9SoroKMTKtZNJAcwjZLJK8kRMcreRYKB',
  status: 'ok',
  value: {
    id: [ 'u11897627', 'u11897627@d93319890' ],
    frozen: null,
    name: '',
    domain: 'd93319890',
    description: '',
    multisig: 1,
    a_txno: 701650,
    c_txno: 701648,
    num_transactions: 1,
    last_active: 1753420486559,
    created_at: 1753420484707,
    wallets: [
      'eetYxbxipRU6o2DTfLvGZ3cth4WJZh',
      'rUmxPNLuDAtJb67LT8XNcidz6y82Bk'
    ],
    callable_to: [ 'all' ],
    disclosed_to: [],
    groups: [],
    managed: false
  }
}


確認のため、新しいウォレットを使ってトランザクションを発行し、そのcallerがnewUserIdと一致していることを確かめます。

In [15]:
var resp = await rpc.call(anotherUserWallet, 'c1update', { id: newUserId, prop: 'name', value: 'user123' });
var resp = await rpc.call(anotherUserWallet, 'c1query', { type: 'a_transaction', id: resp.txid });
if (resp.value.caller[0] === newUserId) console.log('OK'); else console.log('NG');
console.log(resp);

OK
{
  txid: 'x3ngQzxi6itHzCwpyzDF67fRePHStHqVyRxB25rRjJw33',
  status: 'ok',
  value: {
    blkno: 0,
    time: 1753420489085,
    txid: 'xbPMia8Sn7suoRCtBokNJofSQj2uxgXD3uKcvuR53fn6y',
    addr: 'rUmxPNLuDAtJb67LT8XNcidz6y82Bk',
    caller: [ 'u11897627', 'u11897627@d93319890' ],
    callee: [ 'c1update', 'c1update@default' ],
    status: 'denied',
    pack64: 'BQAAAAEAAAD8AAAAAQAAAAABAAAAAQAAA3siY29udHJhY3QiOiJjMXVwZGF0ZSIsImFyZ3MiOnsiaWQiOiJ1MTE4OTc2MjciLCJwcm9wIjoibmFtZSIsInZhbHVlIjoidXNlcjEyMyJ9LCJhZGRyIjoiclVteFBOTHVEQXRKYjY3TFQ4WE5jaWR6Nnk4MkJrIiwiZGVhZGxpbmUiOjE3NTM0MjA1ODg5NzQsIm9yYWNsZSI6IkpKVWpLUFZnek50OGlQWXIiLCJibG9ja3JlZiI6eyJubyI6NzMyMzAsImhhc2giOiJVeGtsUTBaakRuNzRvSGN5ZmE4VFRNNHpPZkZHcFo0NlhWaGdGSmpHQ0R3PSJ9fXKmfUS7jUbimHoc7ELidLTc/gB7I4P1jzlEmdJb17Mtzv4akyMJtCgkYTnLMwJaRRCPfAMAD7ZLr04CMP2JeuYZiMtiS5TJ91AvsFM2LZVm0cpphmjUyYMflI3VDryIIvfGAHLEJvGBVWD2uMZj2o6U122/gzXe5FJ2oEHzds/eMkns4IS99N0oGQvJWDg+P/alzUxbKw2nChnAjgZHZQFRz38edrGD7X1cpJT6i2zFxnFhN7oWveei9xCQJnEoEE6nuRgFnY

## クリーンアップ
このノートブックの中で作成したユーザは今後使用しないので、削除しておきます。

In [16]:
var resp = await rpc.call(adminWallet, 'c1terminate', { id: newUserId });
console.log(resp);

{
  txno: 701653,
  txid: 'xJMyBvFGc5FbZ6qx3eKCGvFSbbD7SEzV2637UjAd8F6VEB',
  status: 'ok',
  value: 'u11897627'
}
